In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

torch.random.manual_seed(42)
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"

device

device(type='mps')

In [ ]:
import cv2
import os
import clip
from PIL import Image
from tqdm import tqdm

test_dir_path = "data/test/"

model, preprocess = clip.load("ViT-B/32", device=device)
text = clip.tokenize([
    "dirty plate",
    "clean plate"
]).to(device)

similarities = []
file_names = []

for file in tqdm(os.listdir(test_dir_path), desc="Processing images"):
    image = preprocess(Image.open(f"{test_dir_path}/{file}")).unsqueeze(0).to(device)

    with torch.no_grad():
        image_features = model.encode_image(image)
        text_features = model.encode_text(text)

        image_features /= image_features.norm(dim=-1, keepdim=True)
        text_features /= text_features.norm(dim=-1, keepdim=True)

        similarity = (100.0 * image_features @ text_features.T).softmax(dim=-1)
        similarities.append(similarity.cpu().numpy()[0])
        file_names.append(file)

similarities = np.array(similarities)
labels = np.argmax(similarities, axis=1)

Processing images: 100%|██████████| 744/744 [00:13<00:00, 56.74it/s]


In [16]:
df = pd.DataFrame({
    "id": os.listdir(test_dir_path),
    "label": labels
})

df["id"] = df["id"].map(lambda x: x[:x.rfind(".")])
df["label"] = df["label"].map({0: "dirty", 1: "cleaned"})
df.sort_values(by="id", inplace=True)

df.to_csv("data/clip_predictions.csv", index=False)

In [17]:
import os
from PIL import Image
from tqdm import tqdm
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression

# --- 1) build training set (based on folders) ---
train_root = "data/train"
classes = {"dirty": 0, "cleaned": 1}

train_paths = []
for cls_name, y_id in classes.items():
    cls_dir = os.path.join(train_root, cls_name)
    for fn in os.listdir(cls_dir):
        if fn.endswith(".jpg"):
            train_paths.append((os.path.join(cls_dir, fn), y_id))

# --- 2) extract CLIP embeddings (frozen) ---
X, y = [], []
model.eval()

for img_path, label_id in tqdm(train_paths, desc="Extracting train embeddings"):
    image = preprocess(Image.open(img_path)).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(image)  # (1, D)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    X.append(feat.cpu().numpy()[0])  # (D,)
    y.append(label_id)

X = np.stack(X)  # (N, D)
y = np.array(y)

# --- 3) train a classifier on top of embeddings ---
clf = LogisticRegression(max_iter=2000).fit(X, y)

# --- 4) run on test and write submission ---
test_dir_path = "data/test"
out_rows = []

for fn in tqdm(sorted(os.listdir(test_dir_path)), desc="Predicting test"):
    if not fn.endswith(".jpg"):
        continue

    image = preprocess(Image.open(os.path.join(test_dir_path, fn))).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = model.encode_image(image)
        feat = feat / feat.norm(dim=-1, keepdim=True)

    # proba for class 1 ("cleaned")
    proba_cleaned = clf.predict_proba(feat.cpu().numpy())[0, 1]
    pred_id = 1 if proba_cleaned >= 0.5 else 0

    pred_label = "cleaned" if pred_id == 1 else "dirty"
    file_id = fn[:fn.rfind(".")]
    out_rows.append((file_id, pred_label))

import pandas as pd
pd.DataFrame(out_rows, columns=["id", "label"]).sort_values("id").to_csv(
    "data/clip_predictions.csv", index=False
)

Predicting test: 100%|██████████| 744/744 [00:13<00:00, 57.22it/s]
